### Setup

These cells are copied unchanged from Section 1 of `exam2.ipynb`: the imports, the Black–Scholes functions and sample data of §1.4, and the simulation code of §1.6–§1.8. Run them before the Results below.

In [ ]:
from hedging_harness.notebook_views import run
_ = run('MonteCarlo_Accuracy_ComputingCost')


### 2.1 Convergence in $N$

Here we look at the sampling error on its own. Every price in this subsection uses the exact one-step scheme of
§1.8, which has no time-step error. So any gap between the Monte Carlo price $\hat V_N$ and the Black–Scholes
price $V_{\text{BS}}$ of §1.4 comes only from using a finite number of paths $N$. Sections 1.6–1.7 make four
predictions, and we test each one:

| §1 predicts | We measure | Shown in |
|---|---|---|
| $\hat V_N \to V_{\text{BS}}$ as $N$ grows | one seeded run, $N = 10^2$ to $10^6$; selected rows in the table | Table 2.1a, Figure 2.1a |
| the error falls as $N^{-1/2}$ | the root-mean-square error (RMSE) of many independent runs at each $N$ | Figure 2.1b |
| the standard error (SE) of one run is a fair guide to its actual error | the average SE beside the RMSE, and how often the 95% interval contains $V_{\text{BS}}$ | Table 2.1b, Figure 2.1b |
| each extra decimal place costs a hundred times as many paths | the time per path, and the $N$ needed for one cent and a tenth of a cent | Table 2.1c |

The single run uses the same random numbers as §1.10, so its estimate at $N = 100{,}000$ repeats the first result
there.


**Table 2.1a** gives three representative sample sizes from one seeded run. Each uses the
first $N$ of the same $10^6$ draws, so these estimates are dependent. Figure 2.1a shows the full progression
and its 95% intervals, $\hat V_N\pm1.96\,\mathrm{SE}$.


**Table 2.1b** repeats the experiment many times. At each $N$ we make $M$ independent runs and compare their
actual error, the RMSE against $V_{\text{BS}}$, with the average SE that each run reports. We also count how often
the 95% interval contains $V_{\text{BS}}$. We use $M = 2{,}000$ runs
for $N \le 10^4$, which gives a coverage SE of about 0.5 percentage points, and fewer runs at large $N$ to save time.


**Table 2.1c** turns the standard error into a cost. We time a run of $10^6$ paths to get the cost per path.
The payoff standard deviation $\hat s$ comes from the $10^6$-path run, as it would from a small pilot run in
practice. The number of paths needed for a 95% half-width $h$ then follows from $1.96\,\hat s/\sqrt N = h$. As a
check on $\hat s$, the table also gives the exact standard deviation of the discounted payoff, found by
integration as in §1.4.

$$
N = \left(\frac{1.96\,\hat s}{h}\right)^2
$$


**Reading the results.** The estimates approach the Black–Scholes prices as the sample grows
(Figure 2.1a). Repeated runs give fitted error slopes of −0.497 for the call and −0.500 for the put, close
to the predicted $-\tfrac12$ (Figure 2.1b). Their RMSE also agrees closely with the average reported SE.
With only 100 paths, the call's interval covers the benchmark in 93.85% of runs, below the nominal 95%;
at 1,000 paths coverage is 95.1% (Table 2.1b). The normal approximation therefore needs more care at small
sample sizes. Precision remains expensive: a 95% half-width of one cent needs about 8.3 million call paths,
and reducing that width by a factor of ten needs a hundred times as many (Table 2.1c).


### 2.2 Convergence in $\delta t$: strong and weak error

Now we turn to the time step. We run Euler, Milstein and the exact scheme on the same random draws, with
$n = 1, 2, 4, \dots, 512$ steps ($\delta t = T/n$) and $N = 200{,}000$ paths at each $n$. The exact scheme gives
the true $S_T$ on every path, so we can measure both kinds of error of §1.8 directly against it:

| Error | What it measures | §1.8 predicts | Shown in |
|---|---|---|---|
| strong | how far each simulated path ends from the exact one | order $\tfrac12$ for Euler, 1 for Milstein | Figure 2.2a, Table 2.2a |
| weak | the bias in the option price | order 1 for both | Figure 2.2b, Table 2.2a |

The price bias is small next to the sampling noise of a price, so we measure it on the payoff difference $D$
between each scheme and the exact scheme on the same path (common random numbers); the mean of $D$ is the weak
error. We then take up the open question of §1.8, whether Milstein's better paths give better prices, at equal
compute (Table 2.2b) and in the error budget of §1.7 (Table 2.2c, Figure 2.2c).

$$
D = e^{-r(T-t)}\big[f(\hat S_T) - f(S_T)\big]
$$


**Table 2.2a** puts the fitted orders beside the theory of §1.8. Each order is the slope of log error against
$\log\delta t$, fitted from $n = 4$ upwards, since the orders describe small $\delta t$. For the weak errors we keep
only points at least 3 SE away from zero, so that pure noise does not bend the line; the table lists the points
used. The constant $a$ is the one in $|\text{weak error}| \approx a\,\delta t$, which the error budget below needs.
Each fitted value comes with a noise SE, found by refitting 1,000 times with every point jittered by its own
standard error.


**Checks on the error estimates.** For $f(S)=S$, each numerical step has mean growth
$1+r\,\delta t$, so the expected terminal stock price is known:

$$
\mathbb E[\hat S_T]=S_0(1+r\,\delta t)^n.
$$

We compare the simulated forward bias with this expression. With a single step, each scheme's option
price is also a one-dimensional integral over the normal draw, which we evaluate independently with
`quad`. The convergence figures show the full ladder; the checks below test its implementation.


**Table 2.2b** compares the schemes at equal compute. We time each scheme on the same $20{,}000 \times 252$
matrix of draws, and time the drawing of the normals separately, since in practice each scheme needs its own
draws. Using the fits of Table 2.2a, the table then gives the steps $n$ each scheme needs to reach a strong error
of 0.01 and a call bias of 0.001, and the time per path that costs.


**Table 2.2c** checks the error budget at the settings of §1.10: daily steps ($n = 252$) and $N = 100{,}000$
paths, on the same draws as there. So the first column repeats the 252-step prices of §1.10. The last column
divides the time-step bias (the mean of $D$) by the price SE.


**Figure 2.2c** compares allocations of a fixed budget using the measured
call-bias constants $a$ and payoff standard deviation $b=\hat s$. We choose the time step by minimising the
error proxy $a\,\delta t+b/\sqrt N$: the magnitude of the bias plus one standard error. With the one-year
expiry used here, $K_{\text{Compute}}=N/\delta t$, so this proxy becomes
$a\,\delta t+b/\sqrt{K_{\text{Compute}}\delta t}$. Differentiating with respect to $\delta t$ gives

$$
a-\frac{b}{2\sqrt{K_{\text{Compute}}}}\delta t^{-3/2}=0,
\qquad
\delta t^*=\left(\frac{b}{2a}\right)^{2/3}K_{\text{Compute}}^{-1/3},
\qquad N^*=K_{\text{Compute}}\delta t^*.
$$

We round this allocation to a whole number of steps, with a minimum of one. The figure plots the predicted
RMSE, $\sqrt{(a\delta t)^2+b^2/N}$, at that allocation. **The allocation minimises bias plus SE, not RMSE**;
the two criteria give the same asymptotic rate but different constants. At a budget of $10^6$ path-steps, the Euler allocation is six steps and 166,666 paths.
The comparison includes daily Euler
steps and exact one-step simulation, with spot checks against independent runs. The fitted bias model is
an approximation for small time steps, so its coarsest-grid predictions need particular care.


**Reading the results.** Milstein's pathwise convergence is faster: its fitted strong order
is 0.99, against 0.49 for Euler (Table 2.2a). The call-price comparison goes differently. Both fitted weak
orders are close to 1, but Milstein has the larger bias constant at these parameters, 0.452 against 0.119
for Euler. Figures 2.2a–b show why path accuracy and price accuracy need separate comparisons.

At daily steps and $N=100{,}000$, the measured vanilla biases are less than 5% of the price SE (Table 2.2c).
Sampling noise is therefore the main error at those settings. Figure 2.2c shows the benefit of spending
more of a fixed budget on paths: at $10^6$ path-steps, the six-step Euler allocation has predicted RMSE
0.0412, compared with 0.2339 for daily Euler and 0.0147 for exact one-step simulation. The coarsest-budget
spot check deviates from the fitted bias model, so this allocation should be read as an approximation.


### 2.3 Antithetic variates at equal cost


**What we measure, and against what.** Section 1.9 predicts that pairing each draw $\phi$ with its mirror
image $-\phi$ makes the estimate less noisy without simulating more paths. It only works if the two payoffs of
a pair tend to move in opposite directions, that is, if their correlation $\rho$ is negative. Every comparison
keeps the total number of simulated paths $N$ fixed, so plain and antithetic sampling cost the same, and the
antithetic standard error is computed from the $N/2$ pair averages (§1.9). At equal cost, the variance falls by
the factor

$$
\frac{\mathrm{SE}^2_{\text{plain}}}{\mathrm{SE}^2_{\text{antithetic}}} = \frac{1}{1+\rho}.
$$

| Prediction (§1.9) | What we measure | Where |
|---|---|---|
| Pairing preserves each scheme's expected price and reduces sampling noise | Error against Black–Scholes, SE and variance ratio | Table 2.3a |
| Pairing removes most of the error in put–call parity | $\hat C - \hat P$ against $S_0 - Ke^{-rT}$ | Table 2.3b |
| The gain is largest deep in the money; a straddle loses | $\rho$ and the factor across strikes | Figure 2.3a |


**Table 2.3a: plain and antithetic Monte Carlo prices under each scheme, at equal cost.** Each row uses
$N = 100{,}000$ simulated paths (50,000 antithetic pairs) and seed `SEED`. The Black–Scholes benchmark is
$C = 10.4506$, $P = 5.5735$. The variance ratio is $\mathrm{SE}^2_{\text{plain}}/\mathrm{SE}^2_{\text{antithetic}}$ at
the same $N$ (1 for the plain rows). It tells us how many times fewer paths antithetic sampling needs for the
same standard error. Run times are wall-clock seconds and vary from run to run.


**Table 2.3b: put–call parity on the Monte Carlo prices.** Each call and put in Table 2.3a was priced on the
same paths. Parity says $C - P = S_0 - Ke^{-rT} = 4.8771$, and the "parity residual" is how far $\hat C - \hat P$
misses it. Path by path, the call payoff minus the put payoff is simply $S_T - K$, so the residual should equal
the error in the simulated discounted stock price, $e^{-rT}\,\overline{S_T} - S_0$. We estimate that error
separately, with its own standard error, by pricing the stock as a call struck at zero. The last column is the
bias each scheme carries by construction: zero for the exact scheme, and
$S_0\big[(1 + r\,\delta t)^n e^{-rT} - 1\big]$ for Euler and Milstein.


**Figure 2.3a** shows the pair correlation $\rho$ and variance-reduction factor for strikes
from $0.6S_0$ to $1.6S_0$, for a call, a put and a straddle (a call plus a put at the same strike). The points
use 500,000 antithetic pairs under exact one-step simulation; the curves come from numerical integration
against the normal density. Pairing changes the variance constant, so the standard error still falls as
$N^{-1/2}$, at a lower level when $\rho<0$.


**Reading the results.** All sixteen prices in Table 2.3a lie within about two standard errors of
Black–Scholes; the largest misses, 2.00 to 2.04 standard errors, are the three plain 252-step calls. At equal
cost, antithetic pairing cuts the call's standard error from 0.0468 to 0.0330 (a variance ratio of 2.00) and
the put's from 0.0273 to 0.0210 (1.70), with almost the same ratios under every scheme. Put–call parity gains
far more: in Table 2.3b, pairing cuts the standard error of the residual from about 0.064 to 0.0125–0.0128, a
variance factor of 25.1 for the one-step exact scheme. Across strikes (Figure 2.3a), the gain grows deep in the
money, to 23.8 for the call at $K/S_0 = 0.6$ and 83.3 for the put at $K/S_0 = 1.40$. The at-the-money straddle
goes the other way: its factor is 0.61, so pairing makes it *worse* than plain sampling.
